# Tracked model validation, calibration and thresholds

Historical baseline notebooks are preserved under ignored notebooks/executed/history/baseline/.
The baseline reports and artifact remain unchanged. This notebook reuses the four
previously training-CV-tuned pipeline configurations, not historical test metrics.
Five-fold stratified training CV chooses the model. Training-only OOF sigmoid
calibration is compared to raw probabilities on validation using Brier score.
All preprocessing is nested inside calibration folds. The existing holdout was
already evaluated historically: its new final result is not an independent fresh test.
No raw customer records or identifiers are displayed.

The implementation is in src/churn/validation_experiment.py, evaluation.py and tracking.py.
MLflow defaults to local SQLite and filesystem artifacts, with no server required.
Set MLFLOW_TRACKING_URI/MLFLOW_EXPERIMENT_NAME to configure tracking explicitly.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

from churn.validation_experiment import ValidationConfig, run_validation

ROOT = Path.cwd().resolve()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").exists()
config = ValidationConfig.from_environment()
print("Configuration:", config)

## Predeclared validation decisions

Keep all 19 predictors: the historical training-CV L1 selector reduced AP.
Choose the model by maximum training CV AP; choose sigmoid only if its validation
Brier score is strictly smaller. Threshold tables use validation probabilities only.
Default threshold remains 0.5. CHURN_FN_COST/CHURN_FP_COST default to illustrative
equal relative costs (1,1), not company costs. Set CHURN_THRESHOLD_POLICY=scenario
to opt into the validation-grid minimum; ties prefer fewer flagged, then higher threshold.
Validation reuse can overfit calibration/threshold choices. Never retune from test results.

After logging the frozen decision, refit on train+validation, evaluate the final selected
model on test once, log only its test metrics, save the full bundle, retrieve it by
explicit MLflow run ID and verify fingerprint/prediction parity on validation examples.
The calibrated estimator's predict() threshold is not the configurable decision rule;
apply the metadata threshold to predict_proba[:,1].

In [ ]:
import os

from churn.notebook_replay import verified_result

replay_path = os.getenv("CHURN_REPLAY_REPORT")
if replay_path:
    result = verified_result(ROOT, Path(replay_path))
    print("Verification replay: recorded results; no fitting, tuning or test evaluation.")
else:
    result = run_validation(ROOT, config)
REPORTS = ROOT / "reports/milestone3" / result["parent_run_id"]
print("Final artifact:", result["artifact_uri"])
print("Frozen decision:", result["frozen_decision"])
display(pd.DataFrame(result["thresholds"]))

In [ ]:
display(Markdown((REPORTS / "validation_report.md").read_text(encoding="utf-8")))
display(Image(filename=str(REPORTS / "validation_analysis.png")))

## Limitations

The cost minimum is scenario-based, not a proven business optimum. Thresholds learned
on training-only validation probabilities may transfer imperfectly after development-set
refitting. Brier/log loss mix discrimination and calibration; ECE is bin-dependent.
The RF/LR CV near-tie is not evidence of clear superiority. The holdout was previously
seen, and validation is reused for choices. Require independent future/temporal validation,
fairness analysis, approved error costs, dependency locking and a release gate before
production. No FastAPI, Docker deployment or unrelated infrastructure is implemented.

For Milestone 4 regression verification, run scripts/run_notebooks.py --verification. This executes all cells, verifies the saved artifact fingerprint, replays recorded model metrics, and saves separate outputs without training or test evaluation.